# Лабораторна робота 5. Завдання 4 (п. 34–43). Розпізнавання власних цифр збереженою CNN

Ноутбук 5.5. Виконав: Бабин Богдан Олегович, група 443-4

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все».
Код оновлено до API Keras 3 (`keras.Input` замість `input_shape`, формат збереження `.keras`, `keras.utils.load_img`).

## 36. Імпортуємо необхідні бібліотеки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, shutil
import keras
from keras.datasets import mnist
from keras import utils
from google.colab import files
%matplotlib inline

## 37. Завантажуємо збережену мережу з GitHub
Вкажіть посилання на файл моделі у своєму репозиторії (кнопка **Raw** на сторінці файлу).
Якщо посилання не працює — комірка запропонує завантажити файл `mnist_cnn_best.keras` з комп'ютера.

In [ ]:
MODEL_URL = 'https://github.com/MarcoGr11/DL-2026/raw/main/lab5/mnist_cnn_best.keras'

!wget -q -O mnist_cnn_best.keras "$MODEL_URL" || true
if not os.path.exists('mnist_cnn_best.keras') or os.path.getsize('mnist_cnn_best.keras') < 10000:
    print('З GitHub не завантажилось — оберіть файл mnist_cnn_best.keras')
    up = files.upload()
    shutil.move(list(up)[0], 'mnist_cnn_best.keras')
model = keras.models.load_model('mnist_cnn_best.keras')
model.summary()

## 38. Перекомпілюємо модель перед використанням

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

## Перевірка на тестовому наборі MNIST

In [ ]:
(_, _), (X_test, y_test) = mnist.load_data()
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1)).astype('float32') / 255
Y_test = utils.to_categorical(y_test)
scores = model.evaluate(X_test, Y_test, verbose=0)
print('Точність на тестових даних: %.2f%%' % (scores[1] * 100))

In [ ]:
idx = np.random.RandomState(1).choice(len(X_test), 10, replace=False)
preds = model.predict(X_test[idx], verbose=0)
correct = 0
plt.figure(figsize=(15, 2.6))
for i, j in enumerate(idx):
    k, t = int(np.argmax(preds[i])), int(np.argmax(Y_test[j]))
    correct += int(k == t)
    plt.subplot(1, 10, i + 1); plt.imshow(X_test[j].reshape(28, 28), cmap='gray'); plt.axis('off')
    plt.title(f'{k} ({preds[i][k]*100:.0f}%)', color='green' if k == t else 'red')
plt.suptitle(f'Зображення з датасету: вірно {correct} з 10'); plt.show()

## Розпізнавання власних зображень

Напишіть цифри 0–9 від руки темною ручкою/маркером на білому папері (одна цифра на фото), сфотографуйте
і завантажте. **Назвіть файл тією цифрою, що на ньому**: `0.jpg`, `1.jpg`, … `9.jpg` — тоді програма сама порахує вірні відповіді.

Обробка фото (як вимагає завдання): переводимо в масив, **інвертуємо** (в MNIST біла цифра на чорному),
обрізаємо по цифрі, вписуємо в 20×20 і ставимо в центр 28×28 (так підготовлено MNIST), нормалізуємо, додаємо вимір.

## 39. Завантажуємо малюнки

In [ ]:
os.makedirs('my_digits', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join('my_digits', name))
digit_paths = sorted(os.path.join('my_digits', f) for f in os.listdir('my_digits'))
print(len(digit_paths), 'файлів:', [os.path.basename(p) for p in digit_paths])

## 40–41. Переводимо в масив, інвертуємо, нормалізуємо, додаємо вимір

In [ ]:
from PIL import Image, ImageOps, ImageFilter
from scipy import ndimage

def prepare_digit(path):
    """Фото цифри (темна ручка на білому папері) -> 28x28 як у MNIST: біла цифра на чорному фоні, по центру."""
    img = ImageOps.exif_transpose(Image.open(path)).convert('L')       # відтінки сірого
    a = np.array(img).astype('float32')
    # 1) інверсія: в MNIST цифра світла на чорному фоні
    if np.median(a) > 127:
        a = 255 - a
    # 2) прибираємо фон паперу (тіні, нерівне освітлення)
    a = np.clip(a - np.median(a) - 25, 0, None)
    if a.max() == 0:
        return np.zeros((28, 28), 'float32')
    a[a < a.max() * 0.25] = 0
    # 3) обрізаємо по рамці цифри
    ys, xs = np.nonzero(a)
    a = a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    # 4) потовщуємо тонку лінію ручки (у MNIST лінії товсті)
    img = Image.fromarray((a / a.max() * 255).astype('uint8'))
    s = 80 / max(img.size)
    img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.LANCZOS)
    img = img.filter(ImageFilter.MaxFilter(5))
    # 5) вписуємо в квадрат 20x20 зі збереженням пропорцій і ставимо в центр поля 28x28 (як у MNIST)
    s = 20 / max(img.size)
    img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.LANCZOS)
    canvas = np.zeros((28, 28), 'float32')
    b = np.array(img, 'float32')
    top, left = (28 - b.shape[0]) // 2, (28 - b.shape[1]) // 2
    canvas[top:top + b.shape[0], left:left + b.shape[1]] = b
    # 6) центр мас цифри — у центр зображення
    cy, cx = ndimage.center_of_mass(canvas)
    canvas = ndimage.shift(canvas, (14 - cy, 14 - cx), mode='constant')
    # 7) нормалізація 0..1
    return np.clip(canvas / canvas.max(), 0, 1)

def digit_from_name(path):
    """Справжня цифра береться з першого символу назви файлу: 3.jpg, 3_1.png, ..."""
    ch = os.path.basename(path)[0]
    return int(ch) if ch.isdigit() else None

In [ ]:
# Покроково на першому зображенні
x = prepare_digit(digit_paths[0])          # масив 28x28, інвертований і нормалізований
x = x.reshape(1, 28, 28, 1)                # додаємо вимір: модель приймає пакет зображень
print(x.shape)

## 42. Запускаємо розпізнавання

In [ ]:
prediction = model.predict(x)
print('Розпізнана цифра:', np.argmax(prediction), '| вірна:', digit_from_name(digit_paths[0]))

In [ ]:
imgs = np.array([prepare_digit(p) for p in digit_paths])
preds = model.predict(imgs.reshape(len(imgs), 28, 28, 1), verbose=0)
rows, correct, known = [], 0, 0
cols = 5; rows_n = int(np.ceil(len(digit_paths) / cols))
plt.figure(figsize=(cols * 3.4, rows_n * 2.0))
for i, p in enumerate(digit_paths):
    k, t = int(np.argmax(preds[i])), digit_from_name(p)
    ok = None if t is None else (k == t)
    if t is not None:
        known += 1; correct += int(ok)
    color = 'black' if ok is None else ('green' if ok else 'red')
    plt.subplot(rows_n, cols * 2, 2 * i + 1); plt.imshow(ImageOps.exif_transpose(Image.open(p)).convert('L'), cmap='gray'); plt.axis('off')
    plt.title(os.path.basename(p), fontsize=8)
    plt.subplot(rows_n, cols * 2, 2 * i + 2); plt.imshow(imgs[i], cmap='gray'); plt.axis('off')
    plt.title(f'{k} ({preds[i][k]*100:.0f}%)', fontsize=11, color=color)
    rows.append({'файл': os.path.basename(p), 'справжня цифра': t, 'передбачено': k,
                 'впевненість, %': round(float(preds[i][k]) * 100, 1), 'вірно': ok})
plt.suptitle('Власні зображення: оригінал і 28×28 після обробки'); plt.tight_layout(); plt.show()
if known:
    print(f'Власні зображення: вірно {correct} з {known} ({correct/known*100:.0f}%)')
pd.DataFrame(rows)

In [ ]:
!zip -q -r lab5_digits.zip my_digits
files.download('lab5_digits.zip')

## 43. Висновки

*(заповніть після запуску)*

- Точність на тестовому наборі: ...; на 10 зображеннях датасету: ... з 10; на власних цифрах: ... з 10.
- Які цифри плутаються і чому: ...
- Що робили проти перенавчання: Dropout, BatchNormalization, EarlyStopping, крос-валідація.